Sandbox to test model set up

# Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

# Data Loading

In [2]:
from bigquery import *

In [3]:
bq = BigQueryClient()
sql_query = """
            SELECT datetime_utc, latitude, longitude, hwp
            FROM watch_duty.hwp_colorado
            ORDER BY datetime_utc ASC
            """
df = bq.query(sql_query)

In [4]:
df.head()

,datetime_utc,latitude,longitude,hwp
0,2025-07-10 00:00:00+00:00,39.44212,-106.63770,21.888397
1,2025-07-10 00:00:00+00:00,37.86189,-105.64904,36.445175
2,2025-07-10 00:00:00+00:00,37.22373,-102.62327,81.745453
3,2025-07-10 00:00:00+00:00,37.03967,-102.50891,78.429558
4,2025-07-10 00:00:00+00:00,38.04809,-107.11562,17.773609


In [6]:
df.dtypes

datetime_utc    datetime64[us, UTC]
latitude                    float64
longitude                   float64
hwp                         float64
dtype: object

# Data Cleaning

In [7]:
# 1. Make sure datetime is the index (required for resampling)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
df = df.set_index('datetime_utc')

# 2. Resample and Interpolate
# We group by location, force a strict 1-hour ('h') frequency, and 
# use 'linear' interpolation to draw a straight line between the missing points.
df_clean = (
    df.groupby(['latitude', 'longitude'])[['hwp']]
    .resample('h')
    .mean() # This just handles duplicates if there happen to be two entries in the same hour
    .interpolate(method='linear')
    .reset_index()
)

# Feature Engineering

In [8]:
# --- 1. TARGET VARIABLE ---
df_clean['target_max_hwp_next_12h'] = (
    df_clean.groupby(['latitude', 'longitude'])['hwp']
    .apply(lambda x: x.shift(-12).rolling(12, min_periods=1).max())
    .reset_index(level=[0,1], drop=True)
)

# --- 2. PREDICTIVE FEATURES ---

# A. Temporal Features
df_clean['hour_of_day'] = df_clean['datetime_utc'].dt.hour
df_clean['month_of_year'] = df_clean['datetime_utc'].dt.month

# B. Lags (Short and Long Memory)
df_clean['hwp_1h_ago'] = df_clean.groupby(['latitude', 'longitude'])['hwp'].shift(1)
df_clean['hwp_3h_ago'] = df_clean.groupby(['latitude', 'longitude'])['hwp'].shift(3)
df_clean['hwp_12h_ago'] = df_clean.groupby(['latitude', 'longitude'])['hwp'].shift(12)
df_clean['hwp_24h_ago'] = df_clean.groupby(['latitude', 'longitude'])['hwp'].shift(24)

# C. Momentum (Rate of Change)
df_clean['hwp_3h_momentum'] = df_clean['hwp_1h_ago'] - df_clean['hwp_3h_ago']
df_clean['hwp_12h_momentum'] = df_clean['hwp_1h_ago'] - df_clean['hwp_12h_ago']

# D. Rolling Mean (The missing piece!)
df_clean['hwp_24h_rolling_mean'] = (
    df_clean.groupby(['latitude', 'longitude'])['hwp']
    .transform(lambda x: x.rolling(24, min_periods=1).mean())
)

# --- 3. CLEAN UP ---
# Drop the NaNs created by shifting 24 hours back and 12 hours forward
model_data = df_clean.dropna()

# Splitting Data

In [10]:
# --- 4. TRAIN / TEST SPLIT ---
features = [
    'hour_of_day', 'month_of_year', 'hwp', 
    'hwp_1h_ago', 'hwp_3h_ago', 'hwp_12h_ago', 'hwp_24h_ago',
    'hwp_3h_momentum', 'hwp_12h_momentum', 'hwp_24h_rolling_mean'
]

X = model_data[features]
y = model_data['target_max_hwp_next_12h']

split_index = int(len(model_data) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

# Train & Evaluate RF Model

In [ ]:
# --- 5. MODEL TRAINING ---
# Initialize the Random Forest
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)

# Train on the log of the target (using log1p to safely handle zeros)
model.fit(X_train, np.log1p(y_train))

# Predict, then convert BACK from log to normal numbers (using expm1)
log_predictions = model.predict(X_test)
final_predictions = np.expm1(log_predictions)

# Calculate your new MAE
mae = mean_absolute_error(y_test, final_predictions)
print(mae)

# Gradient Boosting

In [ ]:
# --- 1. INITIALIZE THE MODEL ---
# max_iter is similar to n_estimators (how many sequential trees to build)
# We use HistGradientBoosting because it is heavily optimized for large datasets
gb_model = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=42)

# --- 2. TRAIN WITH LOG TRANSFORMATION ---
# We still want to use the log transformation to help the model handle the massive spikes!
# np.log1p safely handles zeros by adding 1 before logging.
print("Training Gradient Boosting model...")
gb_model.fit(X_train, np.log1p(y_train))

# --- 3. PREDICT AND TRANSFORM BACK ---
log_predictions = gb_model.predict(X_test)

# np.expm1 is the exact mathematical reverse of np.log1p
final_gb_predictions = np.expm1(log_predictions)

# --- 4. EVALUATE ---
gb_mae = mean_absolute_error(y_test, final_gb_predictions)
print(f"Gradient Boosting MAE: {gb_mae:.2f}")

Training Gradient Boosting model...
Gradient Boosting MAE: 16.69


# Classification

In [12]:
# --- 1. REVERT TO THE 4-TIER BUCKETS ---
bins_4_tier = [-np.inf, 10, 20, 50, np.inf]
class_labels_4_tier = [0, 1, 2, 3] 

y_train_multi = pd.cut(y_train, bins=bins_4_tier, labels=class_labels_4_tier).astype(int)
y_test_multi = pd.cut(y_test, bins=bins_4_tier, labels=class_labels_4_tier).astype(int)

# --- 2. THE SECRET WEAPON: SAMPLE WEIGHTS ---
# This calculates a specific multiplier for every single row in your training data.
# It makes rare or "difficult" classes mathematically more important during training.
weights = compute_sample_weight(class_weight='balanced', y=y_train_multi)

# --- 3. TRAIN WITH WEIGHTS ---
print("Training Balanced 4-Tier Model...")
clf_model_balanced = HistGradientBoostingClassifier(max_iter=200, random_state=42)

# Notice we pass the weights directly into the .fit() function!
clf_model_balanced.fit(X_train, y_train_multi, sample_weight=weights)

Training Balanced 4-Tier Model...


,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",200
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""category"" are considered to be categorical features. The input must be an object exposing a ``__dataframe__`` method such as pandas or polars DataFrames to use this feature.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing values. All categorical values areconverted to floating point numbers. This means that categorical valuesof 1.0 and 1 are treated as the same category.Read more in the :ref:`User Guide `... versionadded:: 0.24.. versionchanged:: 1.2 Added support for feature names... versionchanged:: 1.4 Added `""from_dtype""` option... versionchanged:: 1.6 The default value changed from `None` to `""from_dtype""`.",'from_dt

In [13]:
balanced_predictions = clf_model_balanced.predict(X_test)
print("\n--- Balanced Classification Report ---")
print(classification_report(y_test_multi, balanced_predictions))


--- Balanced Classification Report ---
              precision    recall  f1-score   support

           0       0.57      0.65      0.60       184
           1       0.11      0.10      0.10       105
           2       0.49      0.20      0.28       197
           3       0.54      0.85      0.66       186

    accuracy                           0.49       672
   macro avg       0.43      0.45      0.41       672
weighted avg       0.47      0.49      0.45       672



# Deployed model on GCP Big Query

Included queries used below as reference.

In [ ]:
# Created a new dataset in GCP for feature engineering

"""
CREATE OR REPLACE VIEW `wids-accenture-song.watch_duty.v_hwp_features` AS
WITH BaseData AS (
    SELECT 
        datetime_utc,
        latitude,
        longitude,
        hwp,
        
        LAG(hwp, 1) OVER(
            PARTITION BY CAST(latitude AS STRING), CAST(longitude AS STRING) 
            ORDER BY datetime_utc
        ) AS hwp_1h_ago,
        
        LAG(hwp, 3) OVER(
            PARTITION BY CAST(latitude AS STRING), CAST(longitude AS STRING) 
            ORDER BY datetime_utc
        ) AS hwp_3h_ago,
        
        LAG(hwp, 24) OVER(
            PARTITION BY CAST(latitude AS STRING), CAST(longitude AS STRING) 
            ORDER BY datetime_utc
        ) AS hwp_24h_ago,
        
        -- Corrected: Using UNIX_SECONDS and explicitly casting to TIMESTAMP
        AVG(hwp) OVER(
            PARTITION BY CAST(latitude AS STRING), CAST(longitude AS STRING) 
            ORDER BY UNIX_SECONDS(TIMESTAMP(datetime_utc)) 
            RANGE BETWEEN 86400 PRECEDING AND CURRENT ROW
        ) AS hwp_24h_rolling_mean,

        -- Corrected: Using UNIX_SECONDS and explicitly casting to TIMESTAMP
        MAX(hwp) OVER(
            PARTITION BY CAST(latitude AS STRING), CAST(longitude AS STRING) 
            ORDER BY UNIX_SECONDS(TIMESTAMP(datetime_utc)) 
            RANGE BETWEEN 3600 FOLLOWING AND 43200 FOLLOWING
        ) AS max_hwp_next_12h
    FROM `wids-accenture-song.watch_duty.hwp_colorado`
)
SELECT 
    *,
    -- Calculate Momentum
    (hwp_1h_ago - hwp_3h_ago) AS hwp_3h_momentum,
    
    -- Create the 4-Tier Target Labels directly in SQL
    CASE 
        WHEN max_hwp_next_12h >= 50 THEN 'Extreme'
        WHEN max_hwp_next_12h >= 20 THEN 'High'
        WHEN max_hwp_next_12h >= 10 THEN 'Elevated'
        ELSE 'Safe'
    END AS target_class
FROM BaseData
WHERE hwp_24h_ago IS NOT NULL;
"""

In [ ]:
# building the model

"""
CREATE OR REPLACE MODEL `wids-accenture-song.watch_duty.hwp_refresh_model`
OPTIONS(
  model_type='BOOSTED_TREE_CLASSIFIER',
  input_label_cols=['target_class'],
  auto_class_weights=TRUE, 
  data_split_method='AUTO_SPLIT'
) AS
SELECT 
  EXTRACT(HOUR FROM datetime_utc) AS hour_of_day,
  EXTRACT(MONTH FROM datetime_utc) AS month_of_year,
  hwp, hwp_1h_ago, hwp_3h_ago, hwp_24h_ago, hwp_3h_momentum, hwp_24h_rolling_mean, target_class
FROM 
  `wids-accenture-song.watch_duty.v_hwp_features`
WHERE datetime_utc < '2025-09-01 00:00:00 UTC';
"""

In [ ]:
# evaluating model

"""
SELECT
  *
FROM
  ML.EVALUATE(
    MODEL `wids-accenture-song.watch_duty.hwp_refresh_model`,
    (
      -- We feed it the September test data it has never seen before!
      SELECT * FROM `your_project.your_dataset.v_hwp_features`
      WHERE datetime_utc >= '2025-09-01 00:00:00 UTC'
    )
  );

SELECT
  *
FROM
  ML.CONFUSION_MATRIX(
    MODEL `your_project.your_dataset.hwp_refresh_model`,
    (
      -- We must provide the exact same engineered features!
      SELECT 
        EXTRACT(HOUR FROM datetime_utc) AS hour_of_day,
        EXTRACT(MONTH FROM datetime_utc) AS month_of_year,
        hwp,
        hwp_1h_ago,
        hwp_3h_ago,
        hwp_24h_ago,
        hwp_3h_momentum,
        hwp_24h_rolling_mean,
        target_class
      FROM `your_project.your_dataset.v_hwp_features`
      WHERE datetime_utc >= '2025-09-01 00:00:00 UTC'
    )
  )
ORDER BY 
  expected_label;
"""

Based on the evaulation we found that the model tended to be more risk adverse in labeling HWP values "safe" but also wasn't conservative enough when identifying "extreme values." For this reason we decided to refresh the data every 3 hours for safe labels and every hour for other labels which still made the pipeline more efficient while prioritizing user safety. Given more time we'd like to further refine this forecast by building in more external parameters.

# Calculating next refresh

In [14]:
def get_next_refresh_time(current_time, predicted_class):
    """
    Takes the current datetime and the model's predicted risk class,
    and returns the exact datetime for the next data refresh.
    """
    # Map the model's predicted class to the specific wait time in hours
    refresh_mapping = {
        0: 24, # Safe
        1: 6,  # Elevated
        2: 3,  # High
        3: 1   # Extreme
    }
    
    # Look up the hours to wait based on the prediction
    hours_to_wait = refresh_mapping[predicted_class]
    
    # Calculate the exact future datetime
    # pd.Timedelta makes it incredibly easy to do datetime math
    next_refresh = current_time + pd.Timedelta(hours=hours_to_wait)
    
    return next_refresh

# --- SIMULATION ---
# Let's pretend it is currently March 30th, 2026 at 7:00 PM PST.
# We just ran our model, and it predicted Class 2 (High Risk).

current_time = pd.to_datetime('2026-03-30 19:00:00')
simulated_prediction = 2 

exact_next_refresh = get_next_refresh_time(current_time, simulated_prediction)

print(f"Current Time: {current_time}")
print(f"Predicted Class: {simulated_prediction} (Refresh in 3 hours)")
print(f"Next Refresh Scheduled For: {exact_next_refresh}")

Current Time: 2026-03-30 19:00:00
Predicted Class: 2 (Refresh in 3 hours)
Next Refresh Scheduled For: 2026-03-30 22:00:00
